In [1]:
%load_ext bigquery_magics

import bigquery_magics
bigquery_magics.context.project = "k-move1-kyungpil"

# 1. PIVOT、UNPIVOT

#### 1) PIVOT

`PIVOT`は、**行の値を列名に変換**する演算子であり、`FROM`句で使用する。

#### [変換前]

| 科目 | 点数 |
|---|---:|
| 国語 | 90 |
| 英語 | 85 |
| 数学 | 100 |

#### [変換後]

| 国語 | 英語 | 数学 |
|---:|---:|---:|
| 90 | 85 | 100 |

#### # CASE式を使用する方法

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT '国語' AS subject, 90 AS score UNION ALL
    SELECT '英語', 85 UNION ALL
    SELECT '数学', 100
)

SELECT
    MAX(CASE WHEN subject = '国語' THEN score END) AS `国語`,
    MAX(CASE WHEN subject = '英語' THEN score END) AS `英語`,
    MAX(CASE WHEN subject = '数学' THEN score END) AS `数学`
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,国語,英語,数学
0,90,85,100


#### # PIVOT演算子を使用する方法 

```sql
PIVOT (
    集約関数
    FOR 基準列 IN (値1, 値2, ...)
)
```

In [3]:
%%bigquery

WITH tmp AS (
    SELECT '国語' AS subject, 90 AS score UNION ALL
    SELECT '英語', 85 UNION ALL
    SELECT '数学', 100
)

SELECT *
FROM tmp
PIVOT (
    MAX(score)
    FOR subject IN ('国語', '英語', '数学')
);

Query is running:   0%|          |

Downloading:   0%|          |

,国語,英語,数学
0,90,85,100


#### 2) UNPIVOT

`UNPIVOT`は、**列名を行の値に変換する**演算子であり、`FROM`句で使用する。

#### [変換前]

| 国語 | 英語 | 数学 |
|---:|---:|---:|
| 90 | 85 | 100 |

#### [変換後]

| subject | score |
|---|---:|
| 国語 | 90 |
| 英語 | 85 |
| 数学 | 100 |

#### # UNION ALLを使用する方法

In [5]:
%%bigquery

WITH tmp AS (
    SELECT
        90 AS kor,
        85 AS eng,
        100 AS math
)
select "国語" as subject, kor as score from tmp
union all
select "英語" as subject, eng as score from tmp
union all
select "数学" as subject, math as score from tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,subject,score
0,国語,90
1,英語,85
2,数学,100


#### # UNPIVOT演算子を使用する方法 

```sql
UNPIVOT (
    値列
    FOR 項目列 IN (元の列1, 元の列2, ...)
)
```

In [9]:
%%bigquery

WITH tmp AS (
    SELECT
        90 AS kor,
        85 AS eng,
        100 AS math
)

SELECT 
    subject,
    score
FROM tmp
UNPIVOT (
    score
    FOR subject IN (
        kor AS '国語',
        eng AS '英語',
        math AS '数学'
    )
);

Query is running:   0%|          |

Downloading:   0%|          |

,subject,score
0,国語,90
1,英語,85
2,数学,100


## 2. PIVOT実習

In [19]:
%%bigquery

SELECT*
FROM test.sales_jp;

Query is running:   0%|          |

Downloading:   0%|          |

,region,product,year,amount
0,ソウル,ノートパソコン,2024,100
1,ソウル,ノートパソコン,2025,120
2,ソウル,モニター,2024,80
3,ソウル,モニター,2025,90
4,プサン,ノートパソコン,2024,70
5,プサン,ノートパソコン,2025,60
6,プサン,モニター,2024,50
7,プサン,モニター,2025,55
8,テジョン,ノートパソコン,2024,40
9,テジョン,ノートパソコン,2025,45


#### 1) GROUP BY

In [20]:
%%bigquery

SELECT
  year, SUM(amount)
FROM test.sales_jp
GROUP BY year;

Query is running:   0%|          |

Downloading:   0%|          |

,year,f0_
0,2024,375
1,2025,408


#### 2）PIVOT句を使用する方法 ― FROM句でPIVOTを実行

In [21]:
%%bigquery

WITH tmp AS (
  SELECT year, amount FROM test.sales
)
SELECT *
FROM tmp PIVOT (SUM(amount)
                FOR year IN (2024, 2025));

Query is running:   0%|          |

Downloading:   0%|          |

,_2024,_2025
0,375,408


#### 3）PIVOT句を使用せず、CASE式で行を列に変換する

In [23]:
%%bigquery

SELECT
    SUM(CASE WHEN year = 2024 THEN amount END) AS _2024,
    SUM(CASE WHEN year = 2025 THEN amount END) AS _2025
FROM test.sales_jp;

Query is running:   0%|          |

Downloading:   0%|          |

,_2024,_2025
0,375,408


## 2. UNPIVOT実習

#### 1）UNPIVOT句を使用

まず、`midterm`と`final`を列として持つデータを作成する。

In [24]:
%%bigquery

WITH score AS (
    SELECT '千尋' AS name, 80 AS midterm, 90 AS final UNION ALL
    SELECT 'ハウル', 75, 85 UNION ALL
    SELECT 'キキ', 95, 100
)

SELECT *
FROM score;

Query is running:   0%|          |

Downloading:   0%|          |

,name,midterm,final
0,千尋,80,90
1,ハウル,75,85
2,キキ,95,100


`midterm`・`final`の列を行に変換する。

In [28]:
%%bigquery

WITH score AS (
    SELECT '千尋' AS name, 80 AS midterm, 90 AS final UNION ALL
    SELECT 'ハウル', 75, 85 UNION ALL
    SELECT 'キキ', 95, 100
)

SELECT *
FROM score
UNPIVOT (
    score
    FOR exam IN (midterm, final)
);

Query is running:   0%|          |

Downloading:   0%|          |

,name,score,exam
0,千尋,80,midterm
1,千尋,90,final
2,ハウル,75,midterm
3,ハウル,85,final
4,キキ,95,midterm
5,キキ,100,final


#### 2）UNPIVOT句を使用せず、UNION ALLで列を行に変換する

In [29]:
%%bigquery

WITH score AS (
    SELECT '千尋' AS name, 80 AS midterm, 90 AS final UNION ALL
    SELECT 'ハウル', 75, 85 UNION ALL
    SELECT 'キキ', 95, 100
)

SELECT
    name,
    midterm AS score,
    'midterm' AS exam
FROM score

UNION ALL

SELECT
    name,
    final AS score,
    'final' AS exam
FROM score;

Query is running:   0%|          |

Downloading:   0%|          |

,name,score,exam
0,千尋,80,midterm
1,ハウル,75,midterm
2,キキ,95,midterm
3,千尋,90,final
4,ハウル,85,final
5,キキ,100,final
